In [ ]:
# Balanced / Unbalanced Assignment (Cloud Infrastructure Scheduling)
# -------------------------------------------------------------------
# Workloads are assigned to compute nodes under different system regimes:
# - Balanced: symmetric 1-to-1 scheduling (idealized system)
# - Unbalanced: overload conditions with capacity-limited compute nodes

from optilab.aliases import Model, GRB, quicksum
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()

# -------------------------------------------------------------------------------
# Problem data (cloud system)
# -------------------------------------------------------------------------------
rng = np.random.default_rng(5)

n_nodes = rng.integers(4, 7)     # compute nodes (agents)
n_jobs = rng.integers(6, 12)     # workloads (tasks)

# -------------------------------------------------------------------------------
# Compute node capacities (only used in unbalanced regime)
# -------------------------------------------------------------------------------
capacity = rng.integers(2, 5, size=n_nodes)

# -------------------------------------------------------------------------------
# Job sizes (resource demand proxy)
# -------------------------------------------------------------------------------
job_size = rng.integers(1, 4, size=n_jobs)

# -------------------------------------------------------------------------------
# Latency / allocation cost (cloud routing penalty)
# -------------------------------------------------------------------------------
cost = rng.integers(1, 20, size=(n_nodes, n_jobs))

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
x = [
    [m.add_binary_var(name=f"x_{i}_{j}") for j in range(n_jobs)]
    for i in range(n_nodes)
]

# -------------------------------------------------------------------------------
# Objective: minimize total system cost
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(
        cost[i][j] * x[i][j]
        for i in range(n_nodes)
        for j in range(n_jobs)
    ),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# REGIME SELECTION
# -------------------------------------------------------------------------------
# Toggle between:
#   "balanced"   → idealized cloud scheduling (1-to-1 structure)
#   "unbalanced" → realistic overloaded system

mode = "unbalanced"

# -------------------------------------------------------------------------------
# Constraints
# -------------------------------------------------------------------------------

if mode == "balanced":

    # Each node gets exactly one job (idealized symmetry)
    for i in range(n_nodes):
        m.add_constraint(
            quicksum(x[i][j] for j in range(n_jobs)) == 1
        )

    # Each job is assigned exactly once
    for j in range(n_jobs):
        m.add_constraint(
            quicksum(x[i][j] for i in range(n_nodes)) == 1
        )

else:

    # Each job must be assigned to exactly one node
    for j in range(n_jobs):
        m.add_constraint(
            quicksum(x[i][j] for i in range(n_nodes)) == 1
        )

    # Node capacity constraints (realistic cloud behavior)
    for i in range(n_nodes):
        m.add_constraint(
            quicksum(job_size[j] * x[i][j] for j in range(n_jobs))
            <= capacity[i]
        )

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
assignment = np.array([
    [x[i][j].x for j in range(n_jobs)]
    for i in range(n_nodes)
])

# -------------------------------------------------------------------------------
# Diagnostics
# -------------------------------------------------------------------------------
node_load = assignment @ job_size

print(f"Backend:        {m.backend_name}")
print(f"Mode:           {mode}")
print(f"Nodes:          {n_nodes}")
print(f"Jobs:           {n_jobs}")
print(f"Total capacity: {capacity.sum()}")
print(f"Total demand:   {job_size.sum()}")
print()
print("Cost matrix:\n", cost)
print()
print("Assignment matrix:\n", assignment)
print(f"Node loads:     {node_load}")
print(f"Node capacity:  {capacity}")
print(f"Total cost:     {float(np.sum(assignment * cost))}")

# -------------------------------------------------------------------------------
# Visualization
# -------------------------------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.imshow(assignment, cmap="Blues")
plt.title(f"Cloud Assignment ({mode.capitalize()} Regime)")
plt.xlabel("Jobs")
plt.ylabel("Compute Nodes")
plt.colorbar(label="Assignment")
plt.show()